In [ ]:
#!pip install pyppeteer
#!pip install tavily-python

In [ ]:
!pip install faiss-cpu>=1.15.0
!pip install langchain-community>=0.4.2
!pip install pypeteer 
!pip install tavily-python

In [3]:
import os
from typing import Dict, Any, List, Annotated, Optional, Literal
from IPython.display import Image, display
from urllib.parse import urlparse
import nest_asyncio
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables.graph import MermaidDrawMethod
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.state import CompiledStateGraph
import nest_asyncio
from pathlib import Path
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import sqlite3

#from tavily import TavilyClient

/tmp/ipykernel_100/1388334047.py:27: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
nest_asyncio.apply()
load_dotenv()

True

In [5]:
API_KEY = os.getenv("VOCAREUM_API_KEY")
VECTOR_DIR = Path.cwd().resolve() / "data" / "vectorstore"
#BASE_DIR = Path(__file__).resolve().parents[2]
BASE_DIR = Path.cwd()
CORE_DB = BASE_DIR / "data" / "core" / "udahub.db"
EXTERNAL_DB = BASE_DIR / "data" / "external" / "cultpass.db"

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    base_url="https://openai.vocareum.com/v1",
    api_key=os.getenv("VOCAREUM_API_KEY")
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url="https://openai.vocareum.com/v1",
    api_key=API_KEY,
)

vectorstore = FAISS.load_local(
    str(VECTOR_DIR),
    embeddings,
    allow_dangerous_deserialization=True,
)

def _connect(db_path: Path):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    return conn

In [6]:
@tool
def search_rag_knowledge_base(query: str, category: Optional[str] = None, top_n: int = 3) -> list[dict]:
    """
    Search the support knowledge base for help articles and policy information.

    The vector database contains semantic-searchable support documents in these categories:
    billing, reservation, technical, subscription, and general.

    Use this tool for how-to, troubleshooting, policy, or product-support questions.
    Do not use it for live customer data like account details, reservations, subscriptions, or tickets.

    Args:
        query: Natural-language search query.
        category: Optional category filter.
        top_n: Maximum number of results to return, capped at 3.

    Returns:
        A list of matching articles with article_id, title, category, tags, and snippet.
    """
    top_n = min(max(top_n, 1), 3)

    results = vectorstore.similarity_search(query, k=8)

    if category:
        results = [
            doc for doc in results
            if doc.metadata.get("category") == category
        ]

    results = results[:top_n]

    output = []
    for doc in results:
        output.append(
            {
                "article_id": doc.metadata.get("article_id"),
                "title": doc.metadata.get("title"),
                "category": doc.metadata.get("category"),
                "tags": doc.metadata.get("tags"),
                "snippet": doc.page_content[:220],
            }
        )

    return output

In [20]:
@tool
def get_account_user_by_external_id(account_id: str, external_user_id: str) -> dict:
    """Look up a UDA-Hub user by account and external user ID."""
    conn = _connect(CORE_DB)
    try:
        cursor = conn.cursor()
        cursor.execute(
            """
            SELECT user_id, account_id, external_user_id, user_name, created_at
            FROM users
            WHERE account_id = ? AND external_user_id = ?
            """,
            (account_id, external_user_id),
        )
        row = cursor.fetchone()

        if not row:
            return {"found": False, "message": "No matching user found."}

        return {"found": True, "user": dict(row)}
    finally:
        conn.close()

@tool
def get_user_subscription(external_user_id: str) -> dict:
    """Get a CultPass user's subscription details."""
    conn = _connect(EXTERNAL_DB)
    try:
        cursor = conn.cursor()
        cursor.execute(
            """
            SELECT
                s.subscription_id,
                s.user_id,
                s.status,
                s.tier,
                s.monthly_quota,
                s.started_at,
                s.ended_at
            FROM subscriptions s
            WHERE s.user_id = ?
            """,
            (external_user_id,),
        )
        row = cursor.fetchone()

        if not row:
            return {"found": False, "message": "No subscription found."}

        return {"found": True, "subscription": dict(row)}
    finally:
        conn.close()

@tool
def get_user_reservations(external_user_id: str) -> dict:
    """Get reservations and linked experience details for a CultPass user."""
    conn = _connect(EXTERNAL_DB)
    try:
        cursor = conn.cursor()
        cursor.execute(
            """
            SELECT
                r.reservation_id,
                r.status AS reservation_status,
                r.created_at AS reserved_at,
                e.experience_id,
                e.title,
                e.description,
                e.location,
                e.when,
                e.slots_available,
                e.is_premium
            FROM reservations r
            JOIN experiences e
              ON r.experience_id = e.experience_id
            WHERE r.user_id = ?
            ORDER BY e.when ASC
            """,
            (external_user_id,),
        )
        rows = cursor.fetchall()

        return {
            "found": len(rows) > 0,
            "reservations": [dict(row) for row in rows],
        }
    finally:
        conn.close()

@tool
def get_ticket_details(ticket_id: str) -> dict:
    """Get ticket metadata and linked user/account info."""
    conn = _connect(CORE_DB)
    try:
        cursor = conn.cursor()
        cursor.execute(
            """
            SELECT
                t.ticket_id,
                t.account_id,
                t.user_id,
                t.channel,
                t.created_at,
                tm.status,
                tm.main_issue_type,
                tm.tags,
                u.external_user_id,
                u.user_name
            FROM tickets t
            LEFT JOIN ticket_metadata tm
              ON t.ticket_id = tm.ticket_id
            JOIN users u
              ON t.user_id = u.user_id
            WHERE t.ticket_id = ?
            """,
            (ticket_id,),
        )
        row = cursor.fetchone()

        if not row:
            return {"found": False, "message": "Ticket not found."}

        return {"found": True, "ticket": dict(row)}
    finally:
        conn.close()

@tool
def get_ticket_messages(ticket_id: str) -> dict:
    """Get the message history for a ticket."""
    conn = _connect(CORE_DB)
    try:
        cursor = conn.cursor()
        cursor.execute(
            """
            SELECT message_id, role, content, created_at
            FROM ticket_messages
            WHERE ticket_id = ?
            ORDER BY created_at ASC
            """,
            (ticket_id,),
        )
        rows = cursor.fetchall()

        return {
            "found": len(rows) > 0,
            "messages": [dict(row) for row in rows],
        }
    finally:
        conn.close()

@tool
def escalate_ticket(ticket_id: str) -> dict:
    """Mark an existing support ticket as escalated."""
    conn = _connect(CORE_DB)
    try:
        cursor = conn.cursor()
        cursor.execute(
            """
            UPDATE ticket_metadata
            SET status = ?
            WHERE ticket_id = ?
            """,
            ("escalated", ticket_id),
        )
        conn.commit()

        if cursor.rowcount == 0:
            return {"success": False, "message": "Ticket not found."}

        return {"success": True, "message": "Ticket escalated."}
    finally:
        conn.close()


agentic_tools = [
    escalate_ticket,
    search_rag_knowledge_base,
    get_account_user_by_external_id,
    get_user_subscription,
    get_user_reservations,
    get_ticket_details,
    get_ticket_messages,
]


In [6]:
class SupportState(MessagesState):
    query: str
    customer_id: str
    ticket_id: Optional[str]
    urgency: Optional[Literal["urgent", "normal", "escalation"]]
    response: Optional[str]
    status: Optional[str]



In [7]:
class UrgencyDataType(BaseModel):
    urgency: Literal['urgent', 'normal', 'escalation'] = Field(default= 'normal', description=("Indicates if the user's query is urgent, normal or needs escalation." ) )

@tool
def urgency_detector( query: str ) -> Dict[str, str]:
    """
    Classify the urgency level of a user's support request.
    This tool analyzes the user's wording, tone, sentiment, and stated impact to assign one of three urgency labels:
    - "urgent": The request suggests immediate attention is needed.
    - "normal": The request can be handled through standard support flow.
    - "escalation": The user explicitly asks to speak with a human or the situation should be handed off to a human representative.
    Args:
        query: The user's support message.
    Returns:
        A dictionary containing the predicted urgency label in the form:
        {"urgency": "<label>"} where <label> is one of
        "urgent", "normal", or "escalation".
    """
    messages = [
        SystemMessage(content="""You are an urgency classification expert for customer support tickets.
                                Analyze the user's request, intent, tone, sentiment, and severity of impact.
                                Classify the query as either 'urgent', 'normal' or 'escalation'. Return only the structured output."""),
        HumanMessage(content=f"Classify the urgency of this user query: {query}."),
    ]
    urgency_llm = llm.with_structured_output(UrgencyDataType)
    result = urgency_llm.invoke(messages)
    return {"urgency": result.urgency}

class DomainDataType(BaseModel):
    domain: Literal['billing', 'reservation', 'technical', 'subscription', 'general'] = Field(default='general', description=("Domain category of the user's query." ) )

@tool
def domain_detector( query: str ) -> Dict[str, str]:
    """
    Classifies the input query into one of the five categories ('billing', 'reservation', 'technical', 'subscription', 'general' )
    Args:
        query: The user's input query.
    Returns:
        A dictionary with the detected domain label.
        Example: {"domain": "technical"}
    """
    messages = [
        SystemMessage(content = f"""You are a domain classification expert for customer support tickets.
                                    Determine which domain best matches the user's query.
                                    Valid categories are: billing, reservation, technical, subscription, general.
                                    - 'billing' : Any query related to billing issues, refunds, payment not working, etc 
                                    - 'reservation' : Any query related to reservations such as reservation inquiries, bookings etc.
                                    - 'technical' : Any query related to login issues, website related issue, sign in/sign out, online registration, etc 
                                    - 'subscription' : Any query related to subscription plans, monthly plan, weekly plan, yearly plan, subscription modifications, pausing subscription, etc
                                    - 'general' : Any query inquiring anything in general which is not covered by other categories falls into this category.
                                    Return only the structured output."""),
        HumanMessage(content=f"Classify the domain of this user query: {query}"),
    ]
    domain_llm = llm.with_structured_output(DomainDataType)
    result = domain_llm.invoke(messages)
    return { "domain" : result.domain }

classifier_agent = create_react_agent(
    name="classifier_agent",
    prompt=SystemMessage(
        content=(
            """ 
            You are a ticket classification agent. 
            Your job is to classify a user query by urgency into 'urgent', 'normal' or 'escalation'

            Use the available tools:
            - urgency_detector
            Return JSON only in this exact format:
            {"urgency": "urgent"}
            """
        )
    ),
    model = llm, 
    tools = [urgency_detector]
)


In [8]:
escalation_agent = create_react_agent(
    name="escalation_agent",
    prompt=SystemMessage(
        content=(
            """
            You are an escalation support agent.
            When a user's issue needs human support or the user insists on talking to real human, use the available tool to mark the ticket as escalated.
            After the escalation succeeds, tell the user:
            A customer support representative will get in touch with you soon via email.
            Do not claim success unless the tool confirms it.
            """
        )
    ), 
    model=llm, 
    tools=agentic_tools
)



In [11]:
general_expert = create_react_agent(
    name="general_expert",
    prompt=SystemMessage(
        content=(
                f"You are General Support Expert at UdaHub. "
                "Handle basic product questions, account setup, and general inquiries. "
                "ALWAYS start with '[GENERAL EXPERT]' and be helpful!"
        )
    ), 
    model=llm,
    tools=agentic_tools,
)
reservation_expert = create_react_agent(
    name="reservation_expert",
    prompt=SystemMessage(
        content=(
                f"You are a Reservation Support Expert at UdaHub." 
                "Handle all the reservation related queries."
                "ALWAYS start with '[RESERVATION EXPERT]' and be helpful!"
        )
    ), 
    model=llm,
    tools=agentic_tools,
)
technical_expert = create_react_agent(
    name="technical_expert",
    prompt=SystemMessage(
        content=(
                f"You are a Technical Support Expert at UdaHub." 
                "Handle all the technical queries" 
                "ALWAYS start with '[TECHNICAL EXPERT]' and be helpful!"
        )
    ),
    model=llm,
    tools=agentic_tools,
)
billing_expert =  create_react_agent(
    name="billing_expert",
    prompt=SystemMessage(
        content=(
                f"You are a Billing Support Expert at UdaHub."
                "Handle all the queries related to billing issues."
                "ALWAYS start with '[BILLING EXPERT]' and be helpful!"
        )
    ), 
    model=llm,
    tools=agentic_tools, 
)
subscription_expert = create_react_agent(
    name="subscription_agent", 
    prompt=SystemMessage(
        content=(
                f"You are a Subscription Support Expert at UdaHub."
                "Handle all the queries related to subscription inquiries" 
                "ALWAYS start with '[SUBSCRIPTION EXPERT]' and be helpful!"
        )
    ),
    model=llm, 
    tools=agentic_tools,
) 

@tool
def route_to_technical_expert(query: str, customer_id: str) -> Dict[str, Any]:
    """Route technical issues to the technical support expert."""
    message = HumanMessage(
        content=f"Customer ID: {customer_id}\nTechnical Issue: {query}"
    )
    result = technical_expert.invoke({"messages": [message]})
    last_message: AIMessage = result["messages"][-1]

    return {
        "task": "technical_support",
        "customer_id": customer_id,
        "issue": query,
        "response": last_message.content,
        "status": "resolved",
    }

@tool
def route_to_billing_expert(query: str, customer_id: str) -> Dict[str, Any]:
    """Route billing issues to the billing support expert."""
    message = HumanMessage(
        content=f"Customer ID: {customer_id}\nBilling Issue: {query}"
    )
    result = billing_expert.invoke({"messages": [message]})
    last_message: AIMessage = result["messages"][-1]

    return {
        "task": "billing_support",
        "customer_id": customer_id,
        "issue": query,
        "response": last_message.content,
        "status": "resolved",
    }

@tool
def route_to_reservation_expert(query: str, customer_id: str) -> Dict[str, Any]:
    """Route reservation issues to the reservation support expert."""
    message = HumanMessage(
        content=f"Customer ID: {customer_id}\nReservation Issue: {query}"
    )
    result = reservation_expert.invoke({"messages": [message]})
    last_message: AIMessage = result["messages"][-1]

    return {
        "task": "reservation_support",
        "customer_id": customer_id,
        "issue": query,
        "response": last_message.content,
        "status": "resolved",
    }

@tool
def route_to_general_expert(query: str, customer_id: str) -> Dict[str, Any]:
    """Route general issues to the general support expert."""
    message = HumanMessage(
        content=f"Customer ID: {customer_id}\nGeneral Issue: {query}"
    )
    result = general_expert.invoke({"messages": [message]})
    last_message: AIMessage = result["messages"][-1]

    return {
        "task": "general_support",
        "customer_id": customer_id,
        "issue": query,
        "response": last_message.content,
        "status": "resolved",
    }

@tool
def route_to_subscription_expert(query: str, customer_id: str) -> Dict[str, Any]:
    """Route subscription issues to the subscription support expert."""
    message = HumanMessage(
        content=f"Customer ID: {customer_id}\nSubscription Issue: {query}"
    )
    result = subscription_expert.invoke({"messages": [message]})
    last_message: AIMessage = result["messages"][-1]

    return {
        "task": "subscription_support",
        "customer_id": customer_id,
        "issue": query,
        "response": last_message.content,
        "status": "resolved",
    }

normal_request_supervisor = create_react_agent(
    name="normal_request_supervisor",
    prompt=SystemMessage(
        content="""
            You are the supervisor for normal-priority customer support requests at UdaHub.

            Your job is to coordinate the request, not solve it directly.

            Follow this process:
            1. Read the user's query and customer_id from the conversation.
            2. First call the domain_detector tool to identify the correct support domain.
            3. Based on the detected domain, call exactly one routing tool:
            - billing -> route_to_billing_expert
            - reservation -> route_to_reservation_expert
            - technical -> route_to_technical_expert
            - subscription -> route_to_subscription_expert
            - general -> route_to_general_expert
            4. Always pass both:
            - query
            - customer_id
            5. After the expert responds, return the expert's final response to the user.

            Important rules:
            - Do not guess the domain without using domain_detector first.
            - Do not answer the support issue yourself if a routing tool can handle it.
            - Do not call multiple routing tools for the same request unless absolutely necessary.
            - Keep the workflow efficient and clean.
            - Your final output should be the routed expert's response.

            You are responsible for accurate delegation for normal support requests.
            """
    ),
    model=llm,
    tools=[
        domain_detector,
        route_to_technical_expert,
        route_to_reservation_expert,
        route_to_general_expert,
        route_to_billing_expert,
        route_to_subscription_expert,
    ],
    checkpointer=MemorySaver()
)


urgent_request_supervisor = create_react_agent(
    name="urgent_request_supervisor",
    prompt=SystemMessage(
        content="""
            You are the supervisor for urgent customer support requests at UdaHub.

            Your job is to coordinate urgent requests quickly and route them to the correct expert as fast as possible.
            You do not solve the issue directly unless no routing tool applies.

            Follow this process:
            1. Read the user's query and customer_id from the conversation.
            2. Immediately call the domain_detector tool to identify the correct support domain.
            3. Based on the detected domain, call exactly one routing tool:
            - billing -> route_to_billing_expert
            - reservation -> route_to_reservation_expert
            - technical -> route_to_technical_expert
            - subscription -> route_to_subscription_expert
            - general -> route_to_general_expert
            4. Always pass both:
            - query
            - customer_id
            5. Return the expert's final response.

            Urgent handling rules:
            - Prioritize speed, clarity, and direct action.
            - Do not guess the domain without using domain_detector first.
            - Do not spend extra turns reasoning if the request is clear.
            - Do not call multiple routing tools unless absolutely necessary.
            - Keep responses concise and action-oriented.
            - Your final output should be the routed expert's response.

            You are responsible for fast and accurate delegation of urgent support requests.
            """
    ),
    model=llm,
    tools=[
        domain_detector,
        route_to_technical_expert,
        route_to_reservation_expert,
        route_to_general_expert,
        route_to_billing_expert,
        route_to_subscription_expert,
    ],
    checkpointer=MemorySaver()
)


In [12]:
# General Support Team (3 agents)
general_agents_pool = [
    create_react_agent(
        name=f"general_agent_{i}",
        prompt=SystemMessage(
            content=(
                f"You are General Support Agent {i} at UdaHub. "
                "Handle basic product questions, account setup, and general inquiries. "
                "ALWAYS start with '[GENERAL SUPPORT]' and be helpful!"
            )
        ),
        model=llm,
        tools=agentic_tools,
    ) for i in range(1, 4)
]

reservation_agents_pool = [
    create_react_agent(
        name=f"reservation_agent_{i}",
        prompt=SystemMessage(
            content=(
                f"You are a Reservation Support Agent {i} at UdaHub." 
                "Handle all the reservation related queries."
                "ALWAYS start with '[RESERVATION AGENT]' and be helpful!"
            )
        ),
        model=llm,
        tools=agentic_tools,
    ) for i in range(1,4)
]

technical_agents_pool = [
    create_react_agent(
        name=f"technical_agent_{i}",
        prompt=SystemMessage(
            content=(
                f"You are a Technical Support Agent {i} at UdaHub." 
                "Handle all the technical queries" 
                "ALWAYS start with '[TECHNICAL AGENT]' and be helpful!"
            )
        ),
        model=llm, 
        tools=agentic_tools,
    ) for i in range(1,4)
]

subscription_agents_pool = [
    create_react_agent(
        name = f"subscription_agent_{i}",
        prompt=SystemMessage(
            content=(
                f"You are a Subscription Support Agent {i} at UdaHub."
                "Handle all the queries related to subscription inquiries" 
                "ALWAYS start with '[SUBSCRIPTION AGENT]' and be helpful!"
            )
        ),
        model=llm,
        tools=agentic_tools,
    ) for i in range(1,4)
]

billing_agents_pool = [
    create_react_agent(
        name=f"billing_agent_{i}",
        prompt=SystemMessage(
            content=(
                f"You are a Billing Support Agent {i} at UdaHub."
                "Handle all the queries related to billing issues."
                "ALWAYS start with '[BILLING AGENT]' and be helpful!"
            )
        ),
        model=llm, 
        tools=agentic_tools,
    ) for i in range(1,4)
]
agent_swarm_map = {
    "general_team" : general_agents_pool, 
    "billing_team" : billing_agents_pool,
    "subscription_team" : subscription_agents_pool,
    "technical_team" : technical_agents_pool, 
    "reservation_team": reservation_agents_pool,
}

class RoundRobinState(MessagesState):
    """State that tracks which agent to call next.""" 
    agent_names: List[str] 
    current_agent_index: int 

def update_index(state: RoundRobinState):
    agent_names = state["agent_names"]
    current_agent_index = state.get("current_agent_index", 0)
    new_agent_index = current_agent_index + 1 
    new_agent_index_in_range = new_agent_index % len(agent_names) 
    return { "current_agent_index" : new_agent_index_in_range }

def route_round_robin(state: RoundRobinState):
    """Route tasks in Round-robin fahsion""" 
    agent_names = state["agent_names"] 
    current_agent_index = state.get("current_agent_index",0)
    current_agent_index_in_range = current_agent_index % len(agent_names) 
    active_agent = agent_names[current_agent_index_in_range] 
    return active_agent

def create_team(name: str, agent_pool: List[CompiledStateGraph]):
    workflow = StateGraph(RoundRobinState)

    workflow.add_node( "update_index", update_index ) 
    for agent in agent_pool: 
        workflow.add_node(agent.name, agent)

    workflow.add_edge(START, "update_index")
    workflow.add_conditional_edges(
        source="update_index", path=route_round_robin, path_map=[agent.name for agent in agent_pool]
    )

    graph = workflow.compile(name=name, checkpointer=MemorySaver())
    return graph 

agent_teams = [
    create_team(team_name, agent_pool) for team_name, agent_pool in agent_swarm_map.items()
]

class UrgentState(MessagesState):
    """State that tracks urgent requests/queries to call next within each team"""
    query: str 
    agent_teams : List[CompiledStateGraph] 
    agent_swarm_map : Dict[str, CompiledStateGraph]
    team_to_call : str 



In [ ]:
def classify_urgency(state: SupportState):
    result = urgency_detector.invoke({"query": state["query"]})
    return {"urgency": result["urgency"]}


def route_by_urgency(state: SupportState):
    return state["urgency"]

def handle_normal(state: SupportState):
    message = HumanMessage(
        content=f"Customer ID: {state['customer_id']}\nUser Query: {state['query']}"
    )
    result = normal_request_supervisor.invoke({"messages": [message]})
    last_message: AIMessage = result["messages"][-1]

    response = last_message.content
    if isinstance(response, list):
        response = str(response)

    return {
        "response": response,
        "status": "resolved",
    }


def handle_urgent(state: SupportState):
    query = state["query"]
    customer_id = state["customer_id"]

    domain_result = domain_detector.invoke({"query": query})

    if isinstance(domain_result, dict):
        domain = domain_result.get("domain", "").strip().lower()
    else:
        domain = str(domain_result).strip().lower()

    team_name_map = {
        "general": "general_team",
        "billing": "billing_team",
        "subscription": "subscription_team",
        "technical": "technical_team",
        "reservation": "reservation_team",
    }

    team_to_call = team_name_map.get(domain, "general_team")

    selected_team = next(
        (team for team in agent_teams if team.name == team_to_call),
        None
    )

    if selected_team is None:
        return {
            "response": f"No team found for domain '{domain}'.",
            "status": "pending",
        }

    message = HumanMessage(
        content=f"Customer ID: {customer_id}\nUrgent issue: {query}"
    )

    result = selected_team.invoke(
        {
            "messages": [message],
            "agent_names": [agent.name for agent in agent_swarm_map[team_to_call]],
            "current_agent_index": state.get("current_agent_index", 0),
        }
    )

    last_message = result["messages"][-1]

    return {
        "response": last_message.content,
        "status": "urgent_handled",
    }



def handle_escalation(state: SupportState):
    if not state.get("ticket_id"):
        return {
            "response": "I can connect you with a human support representative, but I need a valid ticket ID to mark this request as escalated.",
            "status": "pending",
        }

    message = HumanMessage(
        content=(
            f"Customer ID: {state['customer_id']}\n"
            f"Ticket ID: {state['ticket_id']}\n"
            f"User Query: {state['query']}"
        )
    )
    result = escalation_agent.invoke({"messages": [message]})
    last_message: AIMessage = result["messages"][-1]

    response = last_message.content
    if isinstance(response, list):
        response = str(response)

    return {
        "response": response,
        "status": "escalated",
    }

workflow = StateGraph(SupportState)

workflow.add_node("classify_urgency", classify_urgency)
workflow.add_node("handle_normal", handle_normal)
workflow.add_node("handle_urgent", handle_urgent)
workflow.add_node("handle_escalation", handle_escalation)

workflow.add_edge(START, "classify_urgency")

workflow.add_conditional_edges(
    "classify_urgency",
    route_by_urgency,
    {
        "normal": "handle_normal",
        "urgent": "handle_urgent",
        "escalation": "handle_escalation",
    },
)

workflow.add_edge("handle_normal", END)
workflow.add_edge("handle_urgent", END)
workflow.add_edge("handle_escalation", END)

support_graph = workflow.compile(checkpointer=MemorySaver())


In [ ]:
def run_support_query(query: str, customer_id: str, ticket_id: Optional[str] = None):
    result = support_graph.invoke(
        {
            "messages": [HumanMessage(content=query)],
            "query": query,
            "customer_id": customer_id,
            "ticket_id": ticket_id,
        }
    )
    return {
        "urgency": result.get("urgency"),
        "status": result.get("status"),
        "response": result.get("response"),
    }


In [ ]:
run_support_query(
    query="I want to talk to a real human agent right now.",
    customer_id="user_003",
    ticket_id="ticket_003",
)
